In [35]:
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor
from sklearn.tree import export_text
from sklearn.metrics import roc_auc_score
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb

In [36]:
df = pd.read_csv('car_fuel_efficiency_2026.csv')
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [37]:
df = df.fillna(0)
df_full_train, df_test = train_test_split(
    df, test_size=0.2, random_state=1
)

df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)
y_train = df_train.fuel_efficiency_mpg.values
y_val = df_val.fuel_efficiency_mpg.values
y_test = df_test.fuel_efficiency_mpg.values

del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']
train_dict = df_train.to_dict(orient='records')
val_dict = df_val.to_dict(orient='records')
test_dict = df_test.to_dict(orient='records')
dv = DictVectorizer(sparse=True)

X_train = dv.fit_transform(train_dict)
X_val = dv.transform(val_dict)
X_test = dv.transform(test_dict)

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

(6000, 16)
(2000, 16)
(2000, 16)


In [38]:
# Question 1
# Let's train a decision tree regressor to predict the fuel_efficiency_mpg variable.

# Train a model with max_depth=1.
# Which original feature is used for splitting the data? If you use DictVectorizer, map a one-hot feature such as origin=USA back to its original column name.

# 'vehicle_weight'
# 'model_year'
# 'origin'
# 'fuel_type'
dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)

print(export_text(dt, feature_names=list(dv.get_feature_names_out())))

# La respuesta es model_year

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



In [39]:
# Question 2
# Train a random forest regressor with these parameters:

# n_estimators=10
# random_state=1
# n_jobs=-1 (optional - to make training faster)
# What's the RMSE of this model on the validation data?

# 0.1837
# 1.837
# 18.37
# 183.7

rf = RandomForestRegressor(
    n_estimators=10,
    random_state=1,
    n_jobs=-1
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_val)

rmse = mean_squared_error(y_val, y_pred) ** 0.5

print(round(rmse, 4))

1.8332


In [40]:
# Question 3
# Now let's experiment with the n_estimators parameter

# Try these values of this parameter: [10, 50, 100, 150].
# Set random_state to 1.
# Evaluate the model on the validation dataset.
# Which value gives the lowest validation RMSE? Consider 3 decimal places for calculating the answer.

# 10
# 50
# 100
# 150

for n in [10, 50, 100, 150]:
    rf = RandomForestRegressor(
        n_estimators=n,
        random_state=1,
        n_jobs=-1
    )

    rf.fit(X_train, y_train)

    y_pred = rf.predict(X_val)

    rmse = mean_squared_error(y_val, y_pred) ** 0.5

    print(n, round(rmse, 3))

10 1.833
50 1.774
100 1.77
150 1.769


In [41]:
# Question 4
# Let's select the best max_depth:

# Try different values of max_depth: [10, 15, 20, 25]
# For each of these values,
# try different values of n_estimators from [10, 50, 100, 150]
# calculate the mean RMSE
# Fix the random seed: random_state=1
# What's the best max_depth, using the mean RMSE?

# 10
# 15
# 20
# 25

for depth in [10, 15, 20, 25]:

    scores = []

    for n in [10, 50, 100, 150]:

        rf = RandomForestRegressor(
            n_estimators=n,
            max_depth=depth,
            random_state=1,
            n_jobs=-1
        )

        rf.fit(X_train, y_train)

        y_pred = rf.predict(X_val)

        rmse = mean_squared_error(y_val, y_pred) ** 0.5

        scores.append(rmse)

    print(depth, round(np.mean(scores), 3))

10 1.754
15 1.784
20 1.787
25 1.786


In [42]:
# Question 5
# We can extract feature importance information from tree-based models.

# At each step of the decision tree learning algorithm, it finds the best split. When doing it, we can calculate "gain" - the reduction in impurity before and after the split. This gain is quite useful in understanding what are the important features for tree-based models.

# In Scikit-Learn, tree-based models contain this information in the feature_importances_ field.

# For this homework question, we'll find the most important feature:

# Train the model with these parameters:
# n_estimators=10,
# max_depth=20,
# random_state=1,
# n_jobs=-1 (optional)
# Get the feature importance information from this model
# What's the most important feature (among these 4)?

# vehicle_weight
# horsepower
# acceleration
# engine_displacement

rf = RandomForestRegressor(
    n_estimators=10,
    max_depth=20,
    random_state=1,
    n_jobs=-1
)

rf.fit(X_train, y_train)

feature_importance = pd.DataFrame({
    'feature': dv.get_feature_names_out(),
    'importance': rf.feature_importances_
})


features = [
    'vehicle_weight',
    'horsepower',
    'acceleration',
    'engine_displacement'
]

feature_importance[
    feature_importance['feature'].isin(features)
].sort_values(by='importance', ascending=False)

,feature,importance
15,vehicle_weight,0.203313
8,horsepower,0.078271
4,engine_displacement,0.061281
0,acceleration,0.054998


In [ ]:
# Now let's train an XGBoost model! For this question, we'll tune the eta parameter:

# Install XGBoost
# Create DMatrix for train and validation
# Create a watchlist
# Train a model with these parameters for 100 rounds:
# xgb_params = {
#     'eta': 0.3, 
#     'max_depth': 6,
#     'min_child_weight': 1,
    
#     'objective': 'reg:squarederror',
#     'nthread': 8,
    
#     'seed': 1,
#     'verbosity': 1,
# }
# Now change eta from 0.3 to 0.1.

# Which eta leads to the best RMSE score on the validation dataset?

# 0.3
# 0.1
# Both give equal value

features = dv.get_feature_names_out().tolist()

dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features)

xgb_params = {
    'eta': 0.3,
    'max_depth': 6,
    'min_child_weight': 1,
    'objective': 'reg:squarederror',
    'nthread': 8,
    'seed': 1,
    'verbosity': 1,
}

watchlist = [(dtrain, 'train'), (dval, 'val')]

model = xgb.train(
    xgb_params,
    dtrain,
    num_boost_round=100,
    evals=watchlist,
    verbose_eval=False
)

y_pred = model.predict(dval)

rmse_03 = mean_squared_error(y_val, y_pred) ** 0.5

print('eta=0.3:', round(rmse_03, 3))



xgb_params['eta'] = 0.1

model = xgb.train(
    xgb_params,
    dtrain,
    num_boost_round=100,
    evals=watchlist,
    verbose_eval=False
)

y_pred = model.predict(dval)

rmse_01 = mean_squared_error(y_val, y_pred) ** 0.5

print('eta=0.1:', round(rmse_01, 3))


if round(rmse_03, 3) < round(rmse_01, 3):
    print('Best eta: 0.3')
elif round(rmse_01, 3) < round(rmse_03, 3):
    print('Best eta: 0.1')
else:
    print('Both give equal value') 

eta=0.3: 1.826
eta=0.1: 1.725
Best eta: 0.1
